# Naive Bayes for Text Classification

### Naive Bayes for Text Classification on Toy Data using Numpy

NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays.

For more information: https://numpy.org/

In [1]:
# import numpy
import numpy as np

#### Data Preparation

The very first step to build a text classification system is to prepare the dataset and preprocess it, which is not part of the model but highly influence the overall performance. This could include:
- Construct Dataset and split the dataset into training, development and test sets.
- Clean Data: removing stop words, removing offensive phrases ...
- Create Vocabulary

In [2]:
# definite a toy text classification dataset

# create training dataset
train_X = np.array([
    'this is good',
    'this is bad',
    'very bad',
    'very good'
])
train_y = np.array([
    'positive',
    'negative',
    'negative',
    'positive'
])

# create test dataset
test_X = np.array([
    'good',
    'bad',
    'very bad',
    'very good'
])
test_y = np.array([
    'positive',
    'negative',
    'negative',
    'positive'
])

In [3]:
# preprocessing

# preprocess text data
def preprocess(text):
    text = text.lower()
    text = text.split()
    stop_words = ['this', 'is']
    text = [word for word in text if word not in stop_words]
    return text

# create vocabulary
vocab = set()
for doc in train_X:
    words = preprocess(doc)
    vocab.update(words)
vocab = list(vocab)
vocab.sort()

#### Model Training

Three steps to train a multinominal Naive Bayes model:
1. Compute the frequency of each word
2. Compute the priors 
3. Compute the likelihoods with Laplace smoothing

In [4]:
# create frequency matrix
freq_matrix = np.zeros((len(train_X), len(vocab)))
for i, doc in enumerate(train_X):
    words = preprocess(doc)
    for word in words:
        freq_matrix[i, vocab.index(word)] += 1

# calculate class priors
classes, class_counts = np.unique(train_y, return_counts=True)
class_priors = class_counts / len(train_y)

# calculate likelihoods with Laplace smoothing
alpha = 1 # smoothing constant
likelihoods = np.zeros((len(classes), len(vocab)))
for i, c in enumerate(classes):
    docs_in_c = freq_matrix[train_y == c, :]
    total_count = np.sum(docs_in_c) + alpha * len(vocab)
    likelihoods[i, :] = (np.sum(docs_in_c, axis=0) + alpha) / total_count

#### Inference/Test

In [5]:
# predict test set
predictions = []
for doc in test_X:
    words = preprocess(doc)
    likelihood_sums = np.zeros(len(classes))
    for i, c in enumerate(classes):
        for word in words:
            if word in vocab:
                likelihood_sums[i] += np.log(likelihoods[i, vocab.index(word)])
        likelihood_sums[i] += np.log(class_priors[i])
    predicted_class = classes[np.argmax(likelihood_sums)]
    predictions.append(predicted_class)
predictions = np.array(predictions)

# calculate accuracy
accuracy = np.sum(predictions == test_y) / len(test_y)
print('Accuracy:', accuracy)

# calculate precision, recall, and F1 score
true_positives = np.sum((predictions == np.array('positive')) & (test_y == 'positive'))
false_positives = np.sum((predictions == 'positive') & (test_y == 'negative'))
false_negatives = np.sum((predictions == 'negative') & (test_y == 'positive'))

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
f1_score = 2 * precision * recall / (precision + recall)
print('Precision:', precision)
print('Recall:', recall)
print('F1 Score:', f1_score)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


#### Contributions of terms to score 

For a given attribute vector $\vec{x}$ the score of class $c$ is:
$$\hat{p}(C=c|\vec{x}) \propto \hat{p}(c) \prod_{i=1}^M \hat{p}(x_i|c)$$
We commonly use the log version:
$$score(c, \vec{x}) = \log \hat{p}(c) + \sum_{i=1}^M \log \hat{p}(x_i|c)$$
Relative contribution of $X_i = x_i$ to class $c$ compared to $c'$ is defined as:
$$\log \hat{p}(x_i|c) - \log \hat{p}(x_i|c') = \log \frac{\hat{p}(x_i|c)}{\hat{p}(x_i|c')}$$
For example:

In [6]:
def extract_likelihood(w):
    dist = {}
    for i, c in enumerate(classes):
        dist[c] = likelihoods[i, vocab.index(w)]
    return dist

print(extract_likelihood('good'))
print(extract_likelihood('very'))

{'negative': 0.16666666666666666, 'positive': 0.5}
{'negative': 0.3333333333333333, 'positive': 0.3333333333333333}


The contribution of the term 'good' to positive sentiment is:
$$\log \hat{p}(x_i|c) - \log \hat{p}(x_i|c') = \log \frac{0.5}{0.167} = 2.99$$

### Naive Bayes using sklearn

\[Wikipedia\] scikit-learn (formerly scikits.learn and also known as sklearn) is a free software machine learning library for the Python programming language.[3] It features various classification, regression and clustering algorithms including support-vector machines, random forests, gradient boosting, k-means and DBSCAN, and is designed to interoperate with the Python numerical and scientific libraries NumPy and SciPy.

For more information:
1. sklearn: https://scikit-learn.org/stable/
2. Naive Bayes using sklearn: https://scikit-learn.org/stable/modules/naive_bayes.html

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# create training dataset
train_data = pd.DataFrame({
    'text': ['this is good', 'this is bad', 'very bad', 'very good'],
    'label': ['positive', 'negative', 'negative', 'positive']
})

# create test dataset
test_data = pd.DataFrame({
    'text': ['good', 'bad', 'very bad', 'very good'],
    'label': ['positive', 'negative', 'negative', 'positive']
})

# vectorize text
vectorizer = CountVectorizer()
train_X = vectorizer.fit_transform(train_data['text'])
test_X = vectorizer.transform(test_data['text'])

# train model
model = MultinomialNB()
model.fit(train_X, train_data['label'])

# make predictions on test data
predictions = model.predict(test_X)

# calculate accuracy
report = classification_report(test_data['label'], predictions)
print(report)

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         2
    positive       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



### Naive Bayes on Real Text Classification Dataset


In [8]:
from sklearn.datasets import fetch_20newsgroups
from pprint import pprint

# Load the 20 newsgroups dataset
newsgroups_train = fetch_20newsgroups(subset='train')
newsgroups_test = fetch_20newsgroups(subset='test')

In [9]:
pprint(list(newsgroups_train.target_names))

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']


In [10]:
for i in range(10):
    print("[Category]", newsgroups_train.target[i])
    print(newsgroups_train.data[i])
    print("==================================================")

[Category] 7
From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----





[Category] 4
From: guykuo@carson.u.washington.edu (Guy Kuo)
Subject: SI Clock Poll - Final Call
Summary: Final call for SI clock reports
Keywords: SI,acceleration,clock,upgrade
Article-I.D.: shelley.1qvfo9INNc3s
Organization: University of Washington
Lines: 11
NNTP

In [11]:
# Train a Naive Bayes

# Transform the text data into numerical vectors
vectorizer = CountVectorizer(stop_words='english')
train_data = vectorizer.fit_transform(newsgroups_train.data)
test_data = vectorizer.transform(newsgroups_test.data)

# Train a Multinomial Naive Bayes classifier
clf = MultinomialNB()
clf.fit(train_data, newsgroups_train.target)

# Predict the class labels for the test data
predicted = clf.predict(test_data)

In [12]:
# Calculate the precision, recall, and F1-score of the classifier
report = classification_report(newsgroups_test.target, predicted)
print(report)

              precision    recall  f1-score   support

           0       0.80      0.81      0.80       319
           1       0.65      0.80      0.72       389
           2       0.80      0.04      0.08       394
           3       0.55      0.80      0.65       392
           4       0.85      0.79      0.82       385
           5       0.69      0.84      0.76       395
           6       0.89      0.74      0.81       390
           7       0.89      0.92      0.91       396
           8       0.95      0.94      0.95       398
           9       0.95      0.92      0.93       397
          10       0.92      0.97      0.94       399
          11       0.80      0.96      0.87       396
          12       0.79      0.70      0.74       393
          13       0.88      0.87      0.87       396
          14       0.84      0.92      0.88       394
          15       0.81      0.95      0.87       398
          16       0.72      0.93      0.81       364
          17       0.93    